In [ ]:
# ===== APPROACH A — CELL 0 : setup (cache-first) =====
# Input: essentials, img384, ocr_384x2  [+ qwen3vl embeddings / rr_scores যদি আগে বানানো থাকে]
# Accelerator: GPU।  Internet: ON।
import os, glob, json, time, re, gc, warnings, subprocess
from collections import Counter
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

OUT  = '/kaggle/working'
INP  = '/kaggle/input'
LABELS = ['same_figure','same_paper','related_papers','unrelated_papers']
COMBOS = ['image+text','text+text','image+image']
T0 = time.time()

def find(name, isdir=False):
    for r in (INP, OUT):
        for h in glob.glob(f'{r}/**/{name}', recursive=True):
            if os.path.isdir(h) == isdir: return h
    return None

def tlog(*a): print(f'[{(time.time()-T0)/60:5.1f} min]', *a, flush=True)

try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    VRAM = torch.cuda.get_device_properties(0).total_memory/1e9 if HAS_GPU else 0
except Exception:
    HAS_GPU, VRAM = False, 0
NCPU = os.cpu_count() or 2

ESS  = os.path.dirname(find('meta_train.parquet'))
IMGD = find('img384', isdir=True)
tlog('essentials:', ESS, '| images:', IMGD)
tlog(f'GPU={HAS_GPU}  VRAM={VRAM:.0f}GB  CPU={NCPU}')

def nrm(x):
    x = np.asarray(x, dtype=np.float32)
    return x / np.linalg.norm(x, axis=1, keepdims=True).clip(1e-8)
def center(x):
    x = np.asarray(x, dtype=np.float32)
    return nrm(x - x.mean(0, keepdims=True))

mtr = pd.read_parquet(f'{ESS}/meta_train.parquet')
mte = pd.read_parquet(f'{ESS}/meta_test.parquet')
for d in (mtr, mte):
    for c in ['t1','t2','h1','h2']: d[c] = d[c].astype(str)
    d['combo'] = np.where(d.t1 < d.t2, d.t1+'+'+d.t2, d.t2+'+'+d.t1)
mtr['y'] = mtr['label'].astype(str)

IH = pd.read_parquet(f'{ESS}/img_hashes.parquet').hash.astype(str).values
TH = pd.read_parquet(f'{ESS}/txt_hashes.parquet').hash.astype(str).values
II = {h:i for i,h in enumerate(IH)}; TI = {h:i for i,h in enumerate(TH)}
texts = pd.read_parquet(f'{ESS}/texts.parquet'); texts['hash'] = texts['hash'].astype(str)
TXT = texts.set_index('hash').loc[TH, 'text'].fillna('').astype(str).values

p = find('ocr_384x2.parquet') or find('ocr.parquet')
if p:
    o = pd.read_parquet(p); om = dict(zip(o.hash.astype(str), o.ocr.fillna('').astype(str)))
    OCR = np.array([om.get(h,'') for h in IH], dtype=object); OCR_OK = True
    tlog('OCR:', p, '| non-empty', round(float(np.mean([len(s)>0 for s in OCR])), 3))
else:
    OCR = np.array(['']*len(IH), dtype=object); OCR_OK = False
    print('⚠️ OCR নেই — lexical feature দুর্বল হবে')

EMB = {'clip_i': nrm(np.load(f'{ESS}/emb_clip_img.npy')),
       'clip_t': nrm(np.load(f'{ESS}/emb_clip_txt.npy')),
       'sci_t' : center(np.load(f'{ESS}/emb_sci_txt.npy')),
       'dino_i': center(np.load(f'{ESS}/emb_dino_img.npy'))}
ALL = pd.concat([mtr.assign(split='tr'), mte.assign(split='te')], ignore_index=True)
tlog('train', mtr.shape, '| test', mte.shape, '| texts', len(TH), '| images', len(IH))


In [ ]:
# ===== APPROACH A — CELL 1 : Qwen3-VL-Embedding (মূল কাজ) =====
# একটাই model image আর text দুটোই নেয়, cross-modal retrieval-এর জন্য যৌথভাবে trained।
# তাই modality gap গঠনগতভাবেই নেই — ঠিক যে রোগে image+text 0.42-এ আটকে আছে।
#
# API: sentence-transformers (model card-এর অফিশিয়াল পথ)
#   model.encode([{"text": ...}, {"image": "/path.jpg"}])  →  (n, 2048)
import subprocess, sys
subprocess.run(f'{sys.executable} -m pip install -q -U "sentence-transformers>=5" qwen-vl-utils',
               shell=True, timeout=1200)
import torch

MODEL  = 'Qwen/Qwen3-VL-Embedding-2B'      # 8B T4/P100-এ সময়ে ধরবে না — 2B-ই ঠিক
BUDGET = 190 * 60                          # embedding-এ সর্বোচ্চ সময়; পরে বাকি cell চলবে

qi, qt = None, None
pi, pt = find('emb_qvl_img.npy'), find('emb_qvl_txt.npy')
if pi and pt:
    qi, qt = np.load(pi), np.load(pt); tlog('cache থেকে পাওয়া গেল', qi.shape, qt.shape)

if qi is None and HAS_GPU:
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(MODEL, device='cuda',
                                    model_kwargs={'torch_dtype': torch.float16})
        tlog('model loaded')

        # ---- smoke test: ৮টা দিয়ে API ও গতি যাচাই ----
        t = time.time()
        sa = model.encode([{'image': f'{IMGD}/{h}.jpg'} for h in IH[:8]], batch_size=8)
        sb = model.encode([{'text': s[:2000]} for s in TXT[:8]], batch_size=8)
        per = (time.time()-t)/16
        tlog(f'smoke ok | img {sa.shape} txt {sb.shape} | {per:.2f}s/object '
             f'→ ২৯,১৭৯টায় ≈ {per*29179/60:.0f} মিনিট')

        def run(kind, items, bs, path):
            """প্রতি ১০০০-এ checkpoint; session মরলে যেখানে থেমেছিল সেখান থেকে"""
            ck = f'{OUT}/_ck_{kind}.npy'
            done = list(np.load(ck)) if os.path.exists(ck) else []
            if done: tlog(f'{kind}: checkpoint থেকে {len(done)}')
            t0 = time.time()
            for i in range(len(done), len(items), 1000):
                done.extend(model.encode(items[i:i+1000], batch_size=bs, show_progress_bar=False))
                np.save(ck, np.array(done, dtype=np.float32))
                el = time.time()-t0; n = max(len(done)-1, 1)
                tlog(f'  {kind} {len(done)}/{len(items)}  বাকি ~{(len(items)-len(done))*el/n/60:.0f}m')
                if time.time()-t0 > BUDGET:
                    tlog(f'⏱️ সময়সীমা — {kind} {len(done)}/{len(items)} পর্যন্ত'); break
            A = np.array(done, dtype=np.float32)
            if len(A) < len(items):          # অসম্পূর্ণ হলে বাকিটা গড় vector (feature নিরপেক্ষ থাকে)
                A = np.vstack([A, np.tile(A.mean(0), (len(items)-len(A), 1))])
            A = nrm(A); np.save(path, A)
            if os.path.exists(ck): os.remove(ck)
            return A

        qi = run('img', [{'image': f'{IMGD}/{h}.jpg'} for h in IH],  8,  f'{OUT}/emb_qvl_img.npy')
        qt = run('txt', [{'text': s[:4000]} for s in TXT],           16, f'{OUT}/emb_qvl_txt.npy')
        del model; gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print('⚠️ Qwen3-VL ব্যর্থ:', repr(e)[:400])
        qi = qt = None

if qi is not None and qt is not None:
    EMB['qvl_i'], EMB['qvl_t'] = nrm(qi), nrm(qt)
    tlog('Qwen3-VL ready', qi.shape, qt.shape)
else:
    print('⚠️ Qwen3-VL নেই — CLIP/SciNCL/DINO দিয়েই চলবে (plan_c-র সমান ফল আশা করো)')
print('spaces:', {k: v.shape for k, v in EMB.items()})


In [ ]:
# ===== APPROACH A — CELL 2 : GATE (৩০ সেকেন্ড, সিদ্ধান্তমূলক) =====
# CLIP-এ image+text-এর cosine ব্যাপ্তি 0.056 — চারটে class এত সরু ফালিতে চাপা যে
# কোনো classifier ওদের আলাদা করতে পারে না। নতুন space সেটা চওড়া করল কিনা দেখি।
def _idx(D):
    sw = (D.t1 == 'image').values
    a = np.array([TI[h] for h in np.where(sw, D.h2, D.h1)])   # caption
    b = np.array([II[h] for h in np.where(sw, D.h1, D.h2)])   # figure
    return a, b

dit = mtr[mtr.combo == 'image+text'].reset_index(drop=True)
a, b = _idx(dit)
rows = []
for name, (ts, isp) in [('clip', ('clip_t','clip_i')), ('qwen3vl', ('qvl_t','qvl_i'))]:
    if ts not in EMB: continue
    c = (EMB[ts][a] * EMB[isp][b]).sum(1)
    g = pd.Series(c).groupby(dit.y.values).mean()
    rows.append({'space': name, **{k: round(v,3) for k,v in g.items()},
                 'ব্যাপ্তি': round(g.max()-g.min(), 3), 'effect': round((g.max()-g.min())/c.std(), 2)})
print('image+text — class-ভিত্তিক গড় cosine')
print(pd.DataFrame(rows).to_string(index=False))
print('\n→ মাপকাঠি: CLIP-এর effect 1.28। qwen3vl স্পষ্টভাবে ছাড়ালে বাজি কাজ করেছে।')
print('   (ব্যাপ্তি নয়, effect-ই আসল — bridge-এ ব্যাপ্তি ৩.৪গুণ বেড়েও লাভ শূন্য ছিল।)')
print('   এই cell কিছু থামায় না; নিচের cell এমনিতেই চলবে।')


In [ ]:
# ===== APPROACH A — CELL 3 : feature (plan_c-র প্রমাণিত engine + নতুন space) =====
from sklearn.feature_extraction.text import TfidfVectorizer
NT, NI = len(TH), len(IH)

def topk_mean(Q, P, k, bs=2048):
    out = np.zeros(len(Q), np.float32)
    for i in range(0, len(Q), bs):
        S = Q[i:i+bs] @ P.T
        out[i:i+bs] = np.partition(S, -k, axis=1)[:, -k:].mean(1)
    return out

def rank_in_pool(Q, P, iq, ip, bs=2048):
    rk = np.zeros(len(iq), np.float32)
    for i in range(0, len(iq), bs):
        S = Q[iq[i:i+bs]] @ P.T
        s = S[np.arange(S.shape[0]), ip[i:i+bs]]
        rk[i:i+bs] = (S > s[:, None]).sum(1)
    return np.log1p(rk)

def best_match(Q, P, rP, k=5, bs=2048):
    out = np.zeros((len(Q), k), np.int64)
    for i in range(0, len(Q), bs):
        S = 2*(Q[i:i+bs] @ P.T) - rP[None, :]
        out[i:i+bs] = np.argsort(-S, 1)[:, :k]
    return out

RAD = {}
def radius(a_, b_):
    if (a_, b_) not in RAD:
        RAD[(a_, b_)] = topk_mean(EMB[a_], EMB[b_], 11 if a_ == b_ else 10)
    return RAD[(a_, b_)]

XT, XI = ('qvl_t','qvl_i') if 'qvl_t' in EMB else ('clip_t','clip_i')
T2I = best_match(EMB[XT], EMB[XI], radius(XI, XT))
I2T = best_match(EMB[XI], EMB[XT], radius(XT, XI))
tlog('2-hop proxy:', XT, XI)

DOCS = np.concatenate([TXT, OCR]).astype(str)          # text j → j ; image i → NT+i
LW = TfidfVectorizer(token_pattern=r'(?u)\b[\w\-\.\+]*\w\b', lowercase=False,
                     sublinear_tf=True, max_df=0.5).fit_transform(DOCS)
LC = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), sublinear_tf=True,
                     min_df=2, max_features=400000).fit_transform(DOCS)
_tok = re.compile(r'[A-Za-z0-9][\w\-\.\+]*[A-Za-z0-9]|\d')
TOKS = [set(t for t in _tok.findall(s) if len(t) >= 2 or t.isdigit()) for s in DOCS]
DF = Counter(t for s in TOKS[:NT] for t in s)
_idf = lambda t: np.log((NT+1)/(DF.get(t,0)+1))
RARE  = [{t for t in s if DF.get(t,0) <= 10 and len(t) >= 3} for s in TOKS]
ENT   = [{t for t in s if (any(c.isdigit() for c in t) and any(c.isalpha() for c in t))
          or (t.isupper() and len(t) >= 2)} for s in TOKS]
NUM   = [{t for t in s if re.fullmatch(r'\d+(\.\d+)?', t) and len(t) >= 2} for s in TOKS]
LOWER = [{t.lower() for t in s if len(t) >= 4 and t.isalpha()} for s in TOKS]
tlog('lexical index ready')

def lex(da, db, p):
    F = {f'{p}_tfw': np.asarray(LW[da].multiply(LW[db]).sum(1)).ravel(),
         f'{p}_tfc': np.asarray(LC[da].multiply(LC[db]).sum(1)).ravel()}
    rn, rj, en, nn_, ws, wj, lw = ([] for _ in range(7))
    for x, y2 in zip(da, db):
        A_, B_ = RARE[x], RARE[y2]; I = A_ & B_
        rn.append(len(I)); rj.append(len(I)/max(1, len(A_|B_)))
        en.append(len(ENT[x] & ENT[y2])); nn_.append(len(NUM[x] & NUM[y2]))
        TA, TB = TOKS[x], TOKS[y2]; I2 = TA & TB
        si = sum(_idf(t) for t in I2); su = sum(_idf(t) for t in TA|TB)
        ws.append(si); wj.append(si/max(1e-6, su)); lw.append(len(LOWER[x] & LOWER[y2]))
    F.update({f'{p}_rare_n':rn, f'{p}_rare_j':rj, f'{p}_ent_n':en, f'{p}_num_n':nn_,
              f'{p}_idf_sum':ws, f'{p}_idf_j':wj, f'{p}_word_n':lw})
    return F

def sim_bundle(F, name, Ea, Eb, ia, ib, same_mod):
    A_, B_ = EMB[Ea], EMB[Eb]
    c = (A_[ia] * B_[ib]).sum(1)
    ra, rb = radius(Ea, Eb)[ia], radius(Eb, Ea)[ib]
    F[f'{name}_cos'] = c
    F[f'{name}_csls'] = 2*c - ra - rb            # hubness সংশোধন
    k1 = rank_in_pool(A_, B_, ia, ib); k2 = rank_in_pool(B_, A_, ib, ia)
    if same_mod:
        F[f'{name}_rmin'], F[f'{name}_rmax'] = np.minimum(k1,k2), np.maximum(k1,k2)
        F[f'{name}_hmin'], F[f'{name}_hmax'] = np.minimum(ra,rb), np.maximum(ra,rb)
    else:
        F[f'{name}_r12'], F[f'{name}_r21'] = k1, k2
        F[f'{name}_h1'],  F[f'{name}_h2']  = ra, rb

def pair_index(D):
    cb = D.combo.iloc[0]
    if cb == 'image+text':
        sw = (D.t1 == 'image').values
        a_ = np.array([TI[h] for h in np.where(sw, D.h2, D.h1)])
        b_ = np.array([II[h] for h in np.where(sw, D.h1, D.h2)])
        la, lb = np.where(sw, D.len2, D.len1), np.where(sw, D.len1, D.len2)
    elif cb == 'text+text':
        a_ = np.array([TI[h] for h in D.h1]); b_ = np.array([TI[h] for h in D.h2])
        la, lb = D.len1.values, D.len2.values
    else:
        a_ = np.array([II[h] for h in D.h1]); b_ = np.array([II[h] for h in D.h2])
        la, lb = D.len1.values, D.len2.values
    return a_, b_, np.log1p(la.astype(float)), np.log1p(lb.astype(float))

def sym(F, n, x, y): F[f'{n}_min'], F[f'{n}_max'] = np.minimum(x,y), np.maximum(x,y)

# Approach B-র reranker score থাকলে নিজে থেকেই ঢুকে পড়বে
RR = {}
for sp, nm in [('rr_train.npy','tr'), ('rr_test.npy','te')]:
    q = find(sp)
    if q: RR[nm] = np.load(q)
if RR: tlog('reranker score পাওয়া গেছে', {k: v.shape for k, v in RR.items()})

def build_scalar(D):
    cb = D.combo.iloc[0]; a_, b_, la, lb = pair_index(D); F = {}
    if cb == 'image+text':
        for ts, isp in [('clip_t','clip_i'), ('qvl_t','qvl_i')]:
            if ts in EMB: sim_bundle(F, ts[:-2], ts, isp, a_, b_, False)
        if OCR_OK:
            F.update(lex(a_, NT + b_, 'cap_ocr'))
            F['ocr_len'] = np.log1p([len(OCR[i]) for i in b_])
        F['px_dino']  = (EMB['dino_i'][T2I[a_,0]] * EMB['dino_i'][b_]).sum(1)
        F['px_dino5'] = (nrm(EMB['dino_i'][T2I[a_]].mean(1)) * EMB['dino_i'][b_]).sum(1)
        F['px_sci']   = (EMB['sci_t'][I2T[b_,0]] * EMB['sci_t'][a_]).sum(1)
        F['px_sci5']  = (nrm(EMB['sci_t'][I2T[b_]].mean(1)) * EMB['sci_t'][a_]).sum(1)
        F.update(lex(I2T[b_,0], a_, 'px_cap'))
        F['self_t'] = (T2I[a_] == b_[:,None]).any(1).astype(np.float32)
        F['self_i'] = (I2T[b_] == a_[:,None]).any(1).astype(np.float32)
        F['len_t'], F['len_i'] = la, lb
        if RR:
            s = np.concatenate([RR['tr'], RR['te']]) if len(RR) == 2 else None
            if s is not None and len(s) == len(D): F['rr'] = s.astype(np.float32)
    elif cb == 'text+text':
        for sp in ['clip_t','sci_t','qvl_t']:
            if sp in EMB: sim_bundle(F, sp, sp, sp, a_, b_, True)
        F.update(lex(a_, b_, 'cap'))
        F['px_dino']  = (EMB['dino_i'][T2I[a_,0]] * EMB['dino_i'][T2I[b_,0]]).sum(1)
        F['px_dino5'] = (nrm(EMB['dino_i'][T2I[a_]].mean(1)) * nrm(EMB['dino_i'][T2I[b_]].mean(1))).sum(1)
        sym(F, 'len', la, lb)
    else:
        for sp in ['clip_i','dino_i','qvl_i']:
            if sp in EMB: sim_bundle(F, sp, sp, sp, a_, b_, True)
        if OCR_OK:
            F.update(lex(NT + a_, NT + b_, 'ocr'))
            sym(F, 'ocr_len', np.log1p([len(OCR[i]) for i in a_]), np.log1p([len(OCR[i]) for i in b_]))
        F['px_sci']  = (EMB['sci_t'][I2T[a_,0]] * EMB['sci_t'][I2T[b_,0]]).sum(1)
        F['px_sci5'] = (nrm(EMB['sci_t'][I2T[a_]].mean(1)) * nrm(EMB['sci_t'][I2T[b_]].mean(1))).sum(1)
        F.update(lex(I2T[a_,0], I2T[b_,0], 'px_cap'))
        sym(F, 'len', la, lb)
    return pd.DataFrame({k: np.asarray(v, dtype=np.float32) for k, v in F.items()})

FULL = {'image+text':  [('clip_t','clip_i'), ('qvl_t','qvl_i')],
        'text+text':   [('clip_t','clip_t'), ('sci_t','sci_t'), ('qvl_t','qvl_t')],
        'image+image': [('clip_i','clip_i'), ('dino_i','dino_i'), ('qvl_i','qvl_i')]}

def build_full(D):
    a_, b_, _, _ = pair_index(D); B = []
    for sa, sb in FULL[D.combo.iloc[0]]:
        if sa in EMB and sb in EMB:
            e1, e2 = EMB[sa][a_], EMB[sb][b_]
            B += [np.abs(e1-e2), e1*e2]
    return np.hstack(B).astype(np.float32)

# ---------- Procrustes bridge : same_figure জোড়া দিয়ে image→text map ----------
# `same_figure` মানে caption আর ঠিক সেই figure — অর্থাৎ ground-truth cross-modal জোড়া।
# train-এ ১০০০টা আছে। এগুলো দিয়ে image space → text space-এর রৈখিক map শেখা যায়,
# তখন modality gap থাকে না। map অবশ্যই fold-এর ভেতরে fit — নাহলে leakage।
from numpy.linalg import svd
from sklearn.model_selection import StratifiedKFold as _SKF
PCA_DIM = 128     # 768 মাত্রায় ~800 জোড়া under-determined; PCA-তে conditioning ঠিক হয়

# ── মাপা ফল (13 Sep, আসল data, CLIP space, image+text) ──────────────────────
#   ব্যাপ্তি   raw 0.0565 → bridge 0.1900   (৩.৪ গুণ চওড়া)
#   effect    raw 1.28   → bridge 1.27     (অপরিবর্তিত — শুধু scale বেড়েছে)
#   macro-F1  bridge ছাড়া 0.4254 → সহ 0.4215  (−0.004)
# tree monotone rescaling-এ উদাসীন, তাই লাভ শূন্য; ২৫৮টা column যোগ হওয়ায় সামান্য ক্ষতি।
# তাই CLIP space-এ বন্ধ। Qwen3-VL space-এর জ্যামিতি আলাদা — cell 2-এর GATE-এ
# qvl-এর effect যদি CLIP-এর 1.28 ছাড়ায়, তবেই BRIDGE_SPACES-এ 'qvl_t' রেখে দেখো।
BRIDGE_SPACES = []          # যেমন: ['qvl_t'] দিলে শুধু নতুন space-এ চলবে

def _pca(M, k=PCA_DIM):
    mu = M.mean(0, keepdims=True)
    _, _, Vt = svd(M - mu, full_matrices=False)
    return mu, Vt[:k].T.astype(np.float32)

def _procrustes(A_, B_):
    U, _, Vt = svd(A_.T @ B_, full_matrices=False)
    return (U @ Vt).astype(np.float32)

def bridge_block(D):
    """image+text-এর জন্য aligned-space pair feature, 5-fold cross-fit (leakage-মুক্ত)"""
    a_, b_, _, _ = pair_index(D)
    tr = (D.split == 'tr').values
    ytr = D.y.values[tr]
    folds = list(_SKF(5, shuffle=True, random_state=0).split(np.zeros(tr.sum()), ytr))
    tr_pos = np.where(tr)[0]
    outs = []
    for ts, isp in [('clip_t','clip_i'), ('qvl_t','qvl_i')]:
        if ts not in EMB or ts not in BRIDGE_SPACES: continue
        A_all, B_all = EMB[isp][b_], EMB[ts][a_]           # image পাশ → text পাশ
        (muA, PA), (muB, PB) = _pca(A_all), _pca(B_all)    # label ছাড়া, তাই leakage নেই
        Ap, Bp = nrm((A_all-muA) @ PA), nrm((B_all-muB) @ PB)
        F = np.zeros((len(D), 2*PCA_DIM + 2), np.float32)
        Ws = []
        for tri, vai in folds:
            m = ytr[tri] == 'same_figure'
            W = _procrustes(Ap[tr_pos[tri]][m].astype(np.float64),
                            Bp[tr_pos[tri]][m].astype(np.float64))
            Ws.append(W)
            r = tr_pos[vai]
            Am = nrm(Ap[r] @ W)
            F[r] = np.c_[np.abs(Am-Bp[r]), Am*Bp[r],
                         (Am*Bp[r]).sum(1), np.linalg.norm(Am-Bp[r], axis=1)]
        te = ~tr                                            # test-এ পাঁচটা map-এর গড়
        for W in Ws:
            Am = nrm(Ap[te] @ W)
            F[te] += np.c_[np.abs(Am-Bp[te]), Am*Bp[te],
                           (Am*Bp[te]).sum(1), np.linalg.norm(Am-Bp[te], axis=1)] / len(Ws)
        c0 = (Ap*Bp).sum(1); c1 = F[:, -2]
        g0 = pd.Series(c0[tr]).groupby(ytr).mean(); g1 = pd.Series(c1[tr]).groupby(ytr).mean()
        tlog(f'  bridge[{ts[:-2]}] ব্যাপ্তি raw {g0.max()-g0.min():.3f} → aligned {g1.max()-g1.min():.3f}')
        outs.append(F)
    return np.hstack(outs).astype(np.float32) if outs else np.zeros((len(D), 0), np.float32)

DATA = {}
for cb in COMBOS:
    D = ALL[ALL.combo == cb].reset_index(drop=True)
    S, Fu = build_scalar(D), build_full(D)
    if cb == 'image+text':
        Fu = np.hstack([Fu, bridge_block(D)])     # প্রমাণিত feature অক্ষত, bridge শুধু যোগ
    assert np.isfinite(S.values).all() and np.isfinite(Fu).all()
    DATA[cb] = (D, S, Fu)
    tlog(f'{cb}: rows={len(D)} scalar={S.shape[1]} full={Fu.shape[1]}')


In [ ]:
# ===== APPROACH A — CELL 4 : train → submission =====
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

REF = {'image+text': 0.42, 'text+text': 0.63, 'image+image': 0.47}   # plan_c
# local-এ Qwen3-VL দিয়ে মাপা (13 Sep, RTX 4060): 0.5951 / 0.6756 / 0.5206,
# pooled OOF 0.6259। এর কাছাকাছি না এলে কিছু ভুল হয়েছে।
FOLDS = StratifiedKFold(5, shuffle=True, random_state=0)

def lgb_p(K, **kw):
    p = dict(objective='multiclass', num_class=K, n_estimators=700, learning_rate=0.05,
             num_leaves=31, colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
             class_weight='balanced', min_child_samples=20, reg_lambda=1.0,
             max_bin=63, force_col_wise=True, verbose=-1, n_jobs=NCPU)
    p.update(kw); return p

def run_lgb(X, y, Xt, K, seeds=(0,), **kw):
    oof = np.zeros((len(y),K)); te = np.zeros((len(Xt),K))
    for s in seeds:
        for tr, va in FOLDS.split(X, y):
            m = lgb.LGBMClassifier(**lgb_p(K, random_state=s, **kw)).fit(X[tr], y[tr])
            oof[va] += m.predict_proba(X[va])/len(seeds); te += m.predict_proba(Xt)/(5*len(seeds))
    return oof, te

def run_lr(X, y, Xt, K, C=0.05):
    oof = np.zeros((len(y),K)); te = np.zeros((len(Xt),K))
    for tr, va in FOLDS.split(X, y):
        sc = StandardScaler().fit(X[tr])
        m = LogisticRegression(C=C, max_iter=300, class_weight='balanced').fit(sc.transform(X[tr]), y[tr])
        oof[va] = m.predict_proba(sc.transform(X[va])); te += m.predict_proba(sc.transform(Xt))/5
    return oof, te

mf1 = lambda y, P, w=None: f1_score(y, (P*(1 if w is None else w)).argmax(1), average='macro')

def blend_w(oofs, y, step=0.1):
    names = list(oofs); best = (-1, None); grid = np.arange(0, 1+1e-9, step)
    def rec(i, left, cur):
        nonlocal best
        if i == len(names)-1:
            w = cur + [left]
            s = mf1(y, sum(wi*oofs[n] for wi, n in zip(w, names)))
            if s > best[0]: best = (s, {n: round(float(x),2) for n, x in zip(names, w)})
            return
        for g in grid:
            if g <= left+1e-9: rec(i+1, left-g, cur+[g])
    rec(0, 1.0, []); return best

def class_mult(P, y, rounds=10):
    w = np.ones(P.shape[1]); best = mf1(y, P, w)
    for _ in range(rounds):
        moved = False
        for j in range(P.shape[1]):
            for m in (0.7,0.8,0.9,0.95,1.05,1.1,1.25,1.4):
                w2 = w.copy(); w2[j] *= m
                s = mf1(y, P, w2)
                if s > best+1e-5: best, w, moved = s, w2, True
        if not moved: break
    return w, best

sub = pd.DataFrame({'id': mte.id.values})
for c in LABELS: sub[c] = 0
REPORT, SIZES = {}, {}

for cb in COMBOS:
    D, S, Fu = DATA[cb]
    m_ = (D.split == 'tr').values
    d, dt = D[m_].reset_index(drop=True), D[~m_].reset_index(drop=True)
    classes = [c for c in LABELS if c in set(d.y)]; K = len(classes)
    y = d.y.map({c:i for i,c in enumerate(classes)}).values
    Xs, Xst = S.values[m_], S.values[~m_]
    Xa, Xat = np.hstack([Xs, Fu[m_]]), np.hstack([Xst, Fu[~m_]])
    tlog(f'--- {cb}: n={len(y)} K={K} scalar={Xs.shape[1]} all={Xa.shape[1]}')

    oofs, tests = {}, {}
    oofs['lgb_s'], tests['lgb_s'] = run_lgb(Xs, y, Xst, K, seeds=(0,1,2), n_estimators=600,
                                            learning_rate=0.03, num_leaves=15, colsample_bytree=0.7)
    tlog(f'  lgb_scalar {mf1(y, oofs["lgb_s"]):.4f}')
    oofs['lgb_a'], tests['lgb_a'] = run_lgb(Xa, y, Xat, K)
    tlog(f'  lgb_all    {mf1(y, oofs["lgb_a"]):.4f}')
    oofs['lr_a'],  tests['lr_a']  = run_lr(Xa, y, Xat, K)
    tlog(f'  lr_all     {mf1(y, oofs["lr_a"]):.4f}')

    s_bl, wts = blend_w(oofs, y)
    P  = sum(wts[n]*oofs[n]  for n in oofs)
    Pt = sum(wts[n]*tests[n] for n in tests)
    mult, s_fin = class_mult(P, y)

    # ---- cascade: related বনাম unrelated-এর জন্য আলাদা binary model ----
    # multiclass model তিনটে সীমানার মধ্যে আপস করে; সবচেয়ে বড় ক্ষতি এই একটাতেই।
    # শুধু ওই দুই শ্রেণির row-তে একটা binary model train করে সিদ্ধান্তটা ওকে দিই।
    # মাপা (13 Sep, image+image, local): 0.4625 → 0.4694 nested-সৎ  (+0.007)
    s_cas, mult_c = s_fin, None
    if 'related_papers' in classes and 'unrelated_papers' in classes:
        iR, iU = classes.index('related_papers'), classes.index('unrelated_papers')
        msk = np.isin(y, [iR, iU]); posm = np.where(msk)[0]
        yb = (y[msk] == iR).astype(int)
        ob = np.zeros(msk.sum()); obt = np.zeros(len(Xat))
        for tri, vai in StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(msk.sum()), yb):
            bm = lgb.LGBMClassifier(**lgb_p(2, objective='binary', num_class=1)).fit(Xa[posm[tri]], yb[tri])
            ob[vai] = bm.predict_proba(Xa[posm[vai]])[:, 1]
            obt += bm.predict_proba(Xat)[:, 1] / 5
        BIN = np.full(len(y), np.nan); BIN[posm] = ob

        def cascade(Pm, B, w, th):
            pr = Pm.argmax(1).copy()
            amb = np.isin(pr, [iR, iU]) & ~np.isnan(B)
            r, u = Pm[amb, iR], Pm[amb, iU]
            pr[amb] = np.where((1-w)*(r/(r+u+1e-9)) + w*B[amb] > th, iR, iU)
            return pr
        Pm = P*mult
        best = (s_fin, (0.0, 0.5))
        for w in (0.0, 0.2, 0.4, 0.6, 0.8, 1.0):
            for th in (0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7):
                sc = f1_score(y, cascade(Pm, BIN, w, th), average='macro')
                if sc > best[0]: best = (sc, (w, th))
        s_cas, mult_c = best
        tlog(f'  cascade (related/unrelated) {s_fin:.4f} → {s_cas:.4f}  w={mult_c[0]} th={mult_c[1]}')

    # সৎ যাচাই: অর্ধেকে tune, বাকিতে মাপা।
    # ⚠️ এলোমেলো করা বাধ্যতামূলক — CSV label অনুযায়ী sorted, তাই P[:h] নিলে
    #    অর্ধেকে কিছু class-ই থাকে না আর সংখ্যাটা অর্থহীন হয় (0.11 এসেছিল)।
    _rs = np.random.default_rng(0).permutation(len(y)); h = len(y)//2
    wh, _ = class_mult(P[_rs[:h]], y[_rs[:h]])
    honest = mf1(y[_rs[h:]], P[_rs[h:]], wh)
    tlog(f'  blend {wts} → {s_bl:.4f} | +thr {s_fin:.4f} | +cascade {s_cas:.4f} | honest {honest:.4f} '
         f'(plan_c {REF[cb]:.2f}, পার্থক্য {s_cas-REF[cb]:+.3f})')
    print(classification_report(y, (P*mult).argmax(1), target_names=classes, digits=3))
    print(pd.DataFrame(confusion_matrix(y, (P*mult).argmax(1)), index=classes, columns=classes))

    REPORT[cb] = {**{n: round(mf1(y,o),4) for n,o in oofs.items()}, 'blend_w': wts,
                  'blend': round(s_bl,4), 'thr': round(s_fin,4), 'final': round(s_cas,4),
                  'cascade_w': mult_c, 'honest': round(float(honest),4), 'plan_c': REF[cb]}
    SIZES[cb] = len(y)
    tg = cb.replace('+','_')
    np.save(f'{OUT}/A_oof_{tg}.npy',  (P*mult).astype(np.float32))
    np.save(f'{OUT}/A_test_{tg}.npy', (Pt*mult).astype(np.float32))
    json.dump(classes, open(f'{OUT}/A_classes_{tg}.json','w'))

    pos = pd.Index(sub.id).get_indexer(dt.id.values); assert (pos >= 0).all()
    Ptm = Pt*mult
    if mult_c is not None and s_cas > s_fin + 1e-4:
        BINt = obt                                   # test-এ binary model-এর গড় probability
        prt = Ptm.argmax(1).copy()
        amb = np.isin(prt, [iR, iU])
        r, u = Ptm[amb, iR], Ptm[amb, iU]
        w, th = mult_c
        prt[amb] = np.where((1-w)*(r/(r+u+1e-9)) + w*BINt[amb] > th, iR, iU)
        pred = prt
    else:
        pred = Ptm.argmax(1)
    for j, c in enumerate(classes): sub.loc[pos[pred == j], c] = 1

assert len(sub) == len(mte) and sub.id.is_unique
assert (sub[LABELS].sum(1) == 1).all()
sub[['id']+LABELS].to_csv(f'{OUT}/submission.csv', index=False)
tot = sum(SIZES.values())
REPORT['overall']      = round(sum(REPORT[c]['final']*SIZES[c]  for c in SIZES)/tot, 4)
REPORT['overall_thr']  = round(sum(REPORT[c]['thr']*SIZES[c]    for c in SIZES)/tot, 4)
REPORT['overall_honest']= round(sum(REPORT[c]['honest']*SIZES[c] for c in SIZES)/tot, 4)
REPORT['plan_c_LB']    = 0.57
REPORT['spaces']       = sorted(EMB)
json.dump(REPORT, open(f'{OUT}/report_A.json','w'), indent=1, default=str)
print('\n'+'='*60); print(json.dumps(REPORT, indent=1, default=str))
print('\nlabel counts:', sub[LABELS].sum().to_dict())
tlog('done →', f'{OUT}/submission.csv')
